# Begin

Courtesy: https://docs.cleanrl.dev/rl-algorithms/ppo-trxl/

In [2]:
# @launchit.collected

In [3]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter, deque # @launchit.collect
import random
import datetime
import json
import pprint
import re
import uuid
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp
import queue

import lark # @launchit.collect

from tqdm.notebook import tqdm

import numpy as np
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
import torch.multiprocessing as torch_mp
import torch._dynamo as dynamo
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Categorical

import gymnasium as gym
import memory_gym 
import av

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect

import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter, RecursiveMovingAverageFilter
from logging_utils import *
from artifact_registry import *
from torch_utils import *
import launchit 
import optuna_multiprocessing  # @launchit.collect
from hp_utils import *
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement

/home/misha/anaconda3/envs/mine/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


# Init

In [4]:
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()
    
def create_config():
    config = namedtuple('Config', 
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, ' + 
                        'self_fname, self_name, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, exec_mode, is_interactive')(
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        self_fname=None,
        self_name=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf)['jupyter_session']
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
            config = config._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)
    
    config = config._replace(is_interactive=config.exec_mode != ExecMode.LAUNCH_MODULE)    
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    return config

In [5]:
# @launchit.disable_2
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.17_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/17_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/17_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/17_rl',
 'self_fname': '/home/misha/dev/mine/neurolab/17_rl/17c_ppo_trxl_memory_mp_01.ipynb',
 'self_name': '17c_ppo_trxl_memory_mp_01',
 'subproject_name': '17_rl',
 'is_cuda': True,
 'cuda_device': 'cuda',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [6]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN = auto()
    WORKER = auto()

LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    # Launch
    launch_goal: LaunchGoal = lu.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED)
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    # System params
    random_seed: int = None
    torch_deterministic: bool = True
    torch_compile: bool = False

    # Environment params
    env_id: str = None
    envs_count: int = 1 # the number of parallel game environments

    # Transformer-XL Agent params
    trxl_layers_count: int = 3 # number of transformer layers
    trxl_heads_count: int = 4 # number of heads used in multi-head attention
    trxl_d_model: int = 384 # the dimension of the transformer
    trxl_memory_length: int = 119 # the length of TrXL's sliding memory window
    trxl_positional_encoding: str = "absolute" # positional encoding type of the transformer: "", "absolute", "learned"
    
    # RL params
    gamma: float = 0.995 # return discount factor gamma
    gae_lambda: float = 0.95 # lambda for the general advantage estimation
    
    # Training procedure params (PPO related) 
    global_steps_count: int = 1_000_000 # total number of steps 
    rollout_steps_count: int = 512 # how many steps to run in a single policy rolllout
    anneal_steps_count: int = 1 * 512 * 10_000 # anneal steps count for learn rate and entropy coeff
    minibatches_count: int = 8
    epochs_count: int = 3 
    clip_coef: float = 0.1 # the surrogate clipping coefficient
    clip_vloss: bool = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    init_ent_coef: float = 0.0001 # initial coefficient of the entropy
    final_ent_coef: float = 0.000001 # final coefficient of the entropy
    vf_coef: float = 0.5 # coefficient of the value function
    max_grad_norm: float = 0.25 # the maximum norm for the gradient clipping
    target_kl: float = None # e target KL divergence threshold
    norm_adv: bool = False # Toggles advantages normalization
    reconstruction_coef: float = 0.0 # the coefficient of the observation reconstruction loss, if set to 0.0 the reconstruction loss is not used

    # Video params
    capture_video: str = 'every(1000000)' # video capture policy depending on steps

    # Optimization params
    optimizer: str = 'AdamW'
    init_learn_rate: float = 2.75e-4
    final_learn_rate: float = 1.0e-5

    # Worker params
    workers_count: int = 1

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()

# Launch

## LaunchState

In [7]:
@dataclass(slots=True)
class LaunchState:
    mp_ctx: object = None
    env: object = None
    env_observation_space_shape: object = None
    env_action_space_shape: object = None
    env_max_episode_steps_count: int = None
    agent: object = None
    workers: list = None
    capture_video_worker: object = None

    @staticmethod
    def new_artifact_registry(is_real=None):
        is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
        is_real = lu.coalesce(is_real, is_launch)
    
        if not is_real:
            mr = Mock()
            mr.register_model.return_value = 0
            return mr
            
        return ArtifactRegistry(CONFIG.model_group_uri)

    @staticmethod
    def new_summary_writer(log_dir, is_real=None):
        is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
        is_real = lu.coalesce(is_real, is_launch)
    
        if not is_real:
            sw = Mock()
            sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
            return sw
        
        return RmqSummaryWriter(log_dir)

## Configure

In [8]:
# @launchit.disable
# @launchit.collect
HP.random_seed = 42
HP.torch_deterministic = True
HP.torch_compile = True
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'torch_compile': True,
 'env_id': None,
 'envs_count': 1,
 'trxl_layers_count': 3,
 'trxl_heads_count': 4,
 'trxl_d_model': 384,
 'trxl_memory_length': 119,
 'trxl_positional_encoding': 'absolute',
 'gamma': 0.995,
 'gae_lambda': 0.95,
 'global_steps_count': 1000000,
 'rollout_steps_count': 512,
 'anneal_steps_count': 5120000,
 'minibatches_count': 8,
 'epochs_count': 3,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'init_ent_coef': 0.0001,
 'final_ent_coef': 1e-06,
 'vf_coef': 0.5,
 'max_grad_norm': 0.25,
 'target_kl': None,
 'norm_adv': False,
 'reconstruction_coef': 0.0,
 'capture_video': 'every(1000000)',
 'optimizer': 'AdamW',
 'init_learn_rate': 0.000275,
 'final_learn_rate': 1e-05,
 'workers_count': 1}


## Create

In [9]:
# @launchit.disable_2
LS = LaunchState()
LS.mp_ctx = torch_mp.get_context('spawn') # Spawn is needed for CUDA, fork doesn't work within PyTorch

optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', HP.launch_id)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    random.seed(HP.random_seed)
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if HP.torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

lc = HP.launch_component()
artifact_registry = LS.new_artifact_registry()
artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    
meta = dict(
    optuna_trial_number=getattr(optuna_trial, 'number', None),
    hypers=HP._asdict(), 
    config=CONFIG._asdict(), 
)

with io.StringIO() as b:
    json.dump(meta, b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = lc.name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = LS.new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

Random seed=42
torch.backends.cudnn.deterministic=True
Tensorboard run=17c_ppo_trxl_memory_mp_01/0


<Mock name='mock.add_text()' id='131559520202832'>

# Environment

## create_env

In [9]:
import pygame

def create_env(env_id, video_dir_name=None, random_seed=None, is_auto_reset=False):
    env = gym.make(env_id, render_mode='debug_rgb_array')

    # Shut up mixer on soundless configurations. Otherwise pygame enters endless loop which consumes 100% CPU 
    if pygame.mixer.get_init():
        pygame.mixer.quit()
    
    if video_dir_name is not None:
        env = gym.wrappers.RecordVideo(env, video_dir_name, episode_trigger=lambda episode_id: episode_id == 0)
        
    env = gym.wrappers.RecordEpisodeStatistics(env)

    if is_auto_reset:
        env = gym.wrappers.Autoreset(env)
    
    if random_seed is not None:
        env.action_space.seed(random_seed) # req-d for random sampling from action space when there are multiple envs

    return env

## Configure 

In [10]:
# @launchit.disable
# @launchit.collect
HP.env_id = "MortarMayhem-Grid-v0"
HP.envs_count = 32 # the number of parallel game environments
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'torch_compile': True,
 'env_id': 'MortarMayhem-Grid-v0',
 'envs_count': 32,
 'trxl_layers_count': 3,
 'trxl_heads_count': 4,
 'trxl_d_model': 384,
 'trxl_memory_length': 119,
 'trxl_positional_encoding': 'absolute',
 'gamma': 0.995,
 'gae_lambda': 0.95,
 'global_steps_count': 1000000,
 'rollout_steps_count': 512,
 'anneal_steps_count': 5120000,
 'minibatches_count': 8,
 'epochs_count': 3,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'init_ent_coef': 0.0001,
 'final_ent_coef': 1e-06,
 'vf_coef': 0.5,
 'max_grad_norm': 0.25,
 'target_kl': None,
 'norm_adv': False,
 'reconstruction_coef': 0.0,
 'capture_video': 'every(1000000)',
 'optimizer': 'AdamW',
 'init_learn_rate': 0.000275,
 'final_learn_rate': 1e-05,
 'workers_count': 1}


## Create

In [11]:
# @launchit.disable_2
LS.env = create_env(HP.env_id)
LS.env_observation_space_shape = LS.env.observation_space.shape
LS.env_action_space_shape = (
    (LS.env.action_space.n.item(),)
    if isinstance(LS.env.action_space, gym.spaces.Discrete)
    else tuple(LS.env.action_space.nvec)
)
LS.env_max_episode_steps_count = LS.env.spec.max_episode_steps

if not LS.env_max_episode_steps_count:
    LS.env.reset()  # Memory Gym envs need to be reset before accessing max_episode_steps
    LS.env_max_episode_steps_count = LS.env.get_wrapper_attr('max_episode_steps')

if LS.env_max_episode_steps_count <= 0:
    LS.env_max_episode_steps_count = 1024  # Memory Gym envs have max_episode_steps set to -1

LOG(f'{LS.env.metadata=}')
LOG(f'{LS.env_observation_space_shape=}')
LOG(f'{LS.env_action_space_shape=}')
LOG(f'{LS.env_max_episode_steps_count=}')
assert HP.trxl_memory_length <= LS.env_max_episode_steps_count, (HP.trxl_memory_length, LS.env_max_episode_steps_count)

LS.env.metadata={'render_modes': ['human', 'rgb_array', 'debug_rgb_array'], 'render_fps': 6}
LS.env_observation_space_shape=(84, 84, 3)
LS.env_action_space_shape=(4,)
LS.env_max_episode_steps_count=119


# Agent

## Components

### layer_init

In [12]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    # torch.nn.init.constant_(layer.bias, bias_const)
    return layer

### PositionalEncoding

In [13]:
# dialogs/positional_embedding.ipynb
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, min_timescale=2.0, max_timescale=1e4):
        super().__init__()
        freqs = torch.arange(0, d_model, min_timescale) # e.g. -> [0, 2, 4, ..., 382], len(freqs) = d_model // min_timescale
        inv_freqs = max_timescale ** (-freqs / d_model) # e.g. [1, 0.96, 0.93, ... 0.001]
        self.register_buffer("inv_freqs", inv_freqs) # declare as non trainable parameter but still part of a model (e.g. included in state_dict, obeys to(device), etc.)

    def forward(self, seq_len):
        seq = torch.arange(seq_len - 1, -1, -1.0, device=self.inv_freqs.device) # -> [seq_len-1, seq_len-2, ... 0]
        sinusoidal_inp = einops.rearrange(seq, 'n -> n 1') * einops.rearrange(self.inv_freqs, 'd -> 1 d') # n -> n () == n -> n 1
        pos_emb = torch.cat((sinusoidal_inp.sin(), sinusoidal_inp.cos()), dim=-1)
        return pos_emb # [seq_len, d_model]

### MultiHeadAttention

In [14]:
class MultiHeadAttention(nn.Module):
    """Multi Head Attention without dropout inspired by https://github.com/aladdinpersson/Machine-Learning-Collection"""

    def __init__(self, d_model, heads_count):
        super().__init__()
        self.d_model = d_model
        self.heads_count = heads_count
        self.head_size = d_model // heads_count # aka head_dim, d_head

        assert self.head_size * heads_count == d_model, 'Embedding dimension needs to be divisible by the number of heads'

        self.values = nn.Linear(self.head_size, self.head_size, bias=False)
        self.keys = nn.Linear(self.head_size, self.head_size, bias=False)
        self.queries = nn.Linear(self.head_size, self.head_size, bias=False)
        self.fc_out = nn.Linear(self.heads_count * self.head_size, d_model)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        values = values.reshape(N, value_len, self.heads_count, self.head_size)
        keys = keys.reshape(N, key_len, self.heads_count, self.head_size)
        query = query.reshape(N, query_len, self.heads_count, self.head_size)

        values = self.values(values)  # (N, value_len, heads, head_dim)
        keys = self.keys(keys)  # (N, key_len, heads, head_dim)
        queries = self.queries(query)  # (N, query_len, heads, head_dim)

        # Dot-product
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys]) # (N, heads, query_len, key_len)

        # Mask padded indices so their attention weights become 0
        if mask is not None:
            energy = energy.masked_fill(mask.unsqueeze(1).unsqueeze(1) == 0, float("-1e20"))  # -inf causes NaN

        # Normalize energy values and apply softmax to retrieve the attention scores
        attention = torch.softmax(
            energy / (self.d_model ** (1 / 2)), dim=3
        )  # attention shape: (N, heads, query_len, key_len)

        # Scale values by attention weights
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]) # (N, query_len, heads, head_dim)
        out = out.reshape(N, query_len, self.heads_count * self.head_size)  # (N, query_len, d_model)

        return self.fc_out(out), attention

### TransformerLayer

In [15]:
class TransformerLayer(nn.Module):
    def __init__(self, d_model, heads_count):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, heads_count)
        self.layer_norm_q = nn.LayerNorm(d_model)
        self.norm_kv = nn.LayerNorm(d_model)
        self.layer_norm_attn = nn.LayerNorm(d_model)
        self.fc_projection = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU())

    def forward(self, value, key, query, mask):
        # Pre-layer normalization (post-layer normalization is usually less effective)
        query_ = self.layer_norm_q(query)
        value = self.norm_kv(value)
        key = value  # K = V -> self-attention
        attention, attention_weights = self.attention(value, key, query_, mask)  # MHA
        x = attention + query  # Skip connection
        x_ = self.layer_norm_attn(x)  # Pre-layer normalization
        forward = self.fc_projection(x_)  # Forward projection
        out = forward + x  # Skip connection
        return out, attention_weights

### Transformer

In [16]:
class Transformer(nn.Module):
    def __init__(self, layers_count, d_model, heads_count, max_episode_steps, positional_encoding):
        super().__init__()
        self.d_model = d_model
        self.max_episode_steps = max_episode_steps
        self.positional_encoding = positional_encoding
        
        assert positional_encoding in ['absolute', 'learned'], f'Unsupported {positional_encoding=}'
        
        if positional_encoding == 'absolute':
            self.pos_embedding = None # delay creation until frist forward. This is to create on target device
        elif positional_encoding == 'learned':
            self.pos_embedding = nn.Parameter(torch.randn(max_episode_steps, d_model))
            
        self.transformer_layers = nn.ModuleList([TransformerLayer(d_model, heads_count) for _ in range(layers_count)])

    def forward(self, x, memories, mask, memory_indices):
        # x.shape = [env, d_model]
        # memories.shape = [env, step, layer, d_model]
        # memory_indices.shape = [env, step]
        
        # Add positional encoding to every transformer layer input
        if self.positional_encoding == 'absolute':
            if self.pos_embedding is None:
                self.pos_embedding = PositionalEncoding(self.d_model)(self.max_episode_steps)
                device = next(iter(self.parameters())).device
                self.pos_embedding = self.pos_embedding.to(device)
            
            pos_embedding = self.pos_embedding[memory_indices]
            pos_embedding = pos_embedding.unsqueeze(2) # shape: [env, step, d_model] -> [env, step, 1, d_model]
            memories = memories + pos_embedding
        elif self.positional_encoding == 'learned':
            memories = memories + self.pos_embedding[memory_indices].unsqueeze(2)

        # Forward transformer layers and return new memories (i.e. hidden states)
        out_memories = []
        
        for i, layer in enumerate(self.transformer_layers):
            out_memories.append(x.detach()) # kms@ Stop-Gradient?
            x, attention_weights = layer(
                value=memories[:, :, i], 
                key=memories[:, :, i], 
                query=x.unsqueeze(1),
                mask=mask
            ) 
            x = x.squeeze() # shape: [env, 1, d_model] -> [env, d_model]
            
            if len(x.shape) == 1:
                x = x.unsqueeze(0)
        
        return x, torch.stack(out_memories, dim=1) # torch.stack output shape: [env, layer, d_model]

## Agent

In [17]:
class Agent(nn.Module):
    @dataclass(slots=True)
    class Params:
        d_model: int = None
        layers_count: int = None
        heads_count: int = None
        positional_encoding: str = None
        observation_space_shape: tuple = None
        action_space_shape: tuple = None
        max_episode_steps_count: int = None
        with_reconstruction_head: bool = None
        
    def __init__(self, params):
        super().__init__()
        self.params = params

        if len(self.params.observation_space_shape) > 1:
            self.encoder = nn.Sequential(
                layer_init(nn.Conv2d(3, 32, 8, stride=4)),
                nn.ReLU(),
                layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                layer_init(nn.Linear(64 * 7 * 7, self.params.d_model)), # kms@ collapse 64 features maps of 7x7 grid (49 elements) to just single vector in embedding space
                nn.ReLU(),
            )
        else:
            self.encoder = layer_init(nn.Linear(observation_space.shape[0], self.params.d_model))

        self.transformer = Transformer(
            self.params.layers_count, 
            self.params.d_model, 
            self.params.heads_count, 
            self.params.max_episode_steps_count, 
            self.params.positional_encoding,
        )

        # kms@ extra MLP layer?
        self.hidden_post_trxl = nn.Sequential(
            layer_init(nn.Linear(self.params.d_model, self.params.d_model)),
            nn.ReLU(),
        )

        # kms@ Create action vectors for multi-descrete actions
        self.actor_branches = nn.ModuleList(
            [
                layer_init(nn.Linear(self.params.d_model, out_features=actions_count), np.sqrt(0.01))
                for actions_count in self.params.action_space_shape
            ]
        )
        self.critic = layer_init(nn.Linear(self.params.d_model, 1), 1)

        if params.with_reconstruction_head:
            self.transposed_cnn = nn.Sequential(
                layer_init(nn.Linear(self.params.d_model, 64 * 7 * 7)),
                nn.ReLU(),
                nn.Unflatten(1, (64, 7, 7)),
                layer_init(nn.ConvTranspose2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                layer_init(nn.ConvTranspose2d(64, 32, 4, stride=2)),
                nn.ReLU(),
                layer_init(nn.ConvTranspose2d(32, 3, 8, stride=4)),
                nn.Sigmoid(),
            )

    def get_value(self, x, memory, memory_mask, memory_indices):
        if len(self.params.observation_space_shape) > 1:
            x = self.encoder(x.permute((0, 3, 1, 2)) / 255.0) # (batch, color, height, width), planar layout
        else:
            x = self.encoder(x)
            
        x, _ = self.transformer(x, memory, memory_mask, memory_indices)
        x = self.hidden_post_trxl(x) # kms@ !!!
        return self.critic(x).flatten()

    ForwardResult = namedtuple('ForwardResult', 'actions, action_log_probs, prob_entropies, values, memories')

    def forward(self, x, memory, memory_mask, memory_indices, action=None):
        if len(self.params.observation_space_shape) > 1:
            x = self.encoder(x.permute((0, 3, 1, 2)) / 255.0) # (batch, color, height, width), planar layout
        else:
            x = self.encoder(x)

        # Here we have an observation packed into a single token mapped to embedding space
        x, memory = self.transformer(x, memory, memory_mask, memory_indices)
        x = self.hidden_post_trxl(x) # kms@ !!!
        
        self.x = x
        probs = [Categorical(logits=branch(x)) for branch in self.actor_branches]
        
        if action is None:
            action = torch.stack([dist.sample() for dist in probs], dim=1)
            
        log_probs = []
        
        for i, dist in enumerate(probs):
            log_probs.append(dist.log_prob(action[:, i]))
            
        entropies = torch.stack([dist.entropy() for dist in probs], dim=1).sum(1).reshape(-1)
        return Agent.ForwardResult(
            actions=action,
            action_log_probs=torch.stack(log_probs, dim=1),
            prob_entropies=entropies,
            values=self.critic(x).flatten(),
            memories=memory,
        )

    def get_action_and_value(self, x, memory, memory_mask, memory_indices, action=None):
        return self.forward(x, memory, memory_mask, memory_indices, action)

    def reconstruct_observation(self):
        x = self.transposed_cnn(self.x)
        return x.permute((0, 2, 3, 1)) # (batch, height, width, color), interleaved (RGB) layout

## Test

### Create test

In [18]:
# @launchit.disable
test_device = 'cpu'
test_device = CONFIG.cuda_device

test_ap = Agent.Params(
    d_model=384,
    layers_count=3,
    heads_count=4,
    positional_encoding='absolute',
    observation_space_shape=(84, 84, 3),
    action_space_shape=(4,),
    max_episode_steps_count=1000,
    with_reconstruction_head=False,
)
test_agent = Agent(test_ap)
test_agent = test_agent.to(test_device)
print(test_agent)
params_count = sum(p.numel() for p in test_agent.parameters())
print(f'{params_count=:_}')

Agent(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=384, bias=True)
    (8): ReLU()
  )
  (transformer): Transformer(
    (transformer_layers): ModuleList(
      (0-2): 3 x TransformerLayer(
        (attention): MultiHeadAttention(
          (values): Linear(in_features=96, out_features=96, bias=False)
          (keys): Linear(in_features=96, out_features=96, bias=False)
          (queries): Linear(in_features=96, out_features=96, bias=False)
          (fc_out): Linear(in_features=384, out_features=384, bias=True)
        )
        (layer_norm_q): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (norm_kv): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (layer_norm_attn):

### Smoke test

In [19]:
# @launchit.disable
test_envs_count = 2
test_steps_count = 10

test_batch = torch.zeros((test_envs_count,) + test_ap.observation_space_shape).to(test_device)
print(f'{test_batch.shape=}')

r = test_agent.get_action_and_value(
    x=test_batch, 
    memory=torch.zeros((test_envs_count, test_steps_count, test_ap.layers_count, test_ap.d_model)).to(test_device),
    memory_mask=None,
    memory_indices=torch.zeros((test_envs_count, test_steps_count), dtype=torch.long).to(test_device),
)

shape = einops.parse_shape(r.actions, 'e a')
assert shape['e'] == test_envs_count
assert shape['a'] == len(test_ap.action_space_shape)
print(f'{r.actions.shape=}, {shape=}')

shape = einops.parse_shape(r.action_log_probs, 'e a')
assert shape['e'] == test_envs_count
assert shape['a'] == len(test_ap.action_space_shape)
print(f'{r.action_log_probs.shape=}, {shape=}')

shape = einops.parse_shape(r.prob_entropies, 'e')
assert shape['e'] == test_envs_count
print(f'{r.prob_entropies.shape=}, {shape=}')

shape = einops.parse_shape(r.values, 'e')
assert shape['e'] == test_envs_count
print(f'{r.values.shape=}, {shape=}')

shape = einops.parse_shape(r.memories, 'e l d')
assert shape['e'] == test_envs_count
assert shape['l'] == test_ap.layers_count
assert shape['d'] == test_ap.d_model
print(f'{r.memories.shape=}, {shape=}')

test_batch.shape=torch.Size([2, 84, 84, 3])
r.actions.shape=torch.Size([2, 1]), shape={'e': 2, 'a': 1}
r.action_log_probs.shape=torch.Size([2, 1]), shape={'e': 2, 'a': 1}
r.prob_entropies.shape=torch.Size([2]), shape={'e': 2}
r.values.shape=torch.Size([2]), shape={'e': 2}
r.memories.shape=torch.Size([2, 3, 384]), shape={'e': 2, 'l': 3, 'd': 384}


### Quick play test

In [20]:
# @launchit.disable
test_env = create_env(HP.env_id, is_auto_reset=True)
test_obs, _ = test_env.reset(seed=HP.random_seed)
test_step = 0
test_hstrace = torch.zeros((LS.env_max_episode_steps_count, HP.trxl_layers_count, HP.trxl_d_model), dtype=torch.float32).to(test_device)
test_causal_masks = torch.tril(torch.ones((HP.trxl_memory_length, HP.trxl_memory_length)), diagonal=-1).to(test_device)
test_hstrace_indices = torch.arange(HP.trxl_memory_length).long().to(test_device)

with eval_guard(test_agent):
    with torch.no_grad():
        for _ in tqdm(range(0, 200)): 
            # test_hstrace_indices update logic: the rightmost index must exceed test_step, otherwise - slide window
            if test_hstrace_indices[-1] < test_step:
                test_hstrace_indices += (test_step - test_hstrace_indices[-1])

            assert test_hstrace_indices[-1] >= test_step

            test_agent_result = test_agent.get_action_and_value(
                x=torch.tensor(test_obs).unsqueeze(0).to(test_device),
                memory=test_hstrace[test_hstrace_indices].unsqueeze(0),
                memory_mask=test_causal_masks[min(test_step, HP.trxl_memory_length - 1)].unsqueeze(0),
                memory_indices=test_hstrace_indices.unsqueeze(0),
            )

            test_action = test_agent_result.actions[0].item()
            test_hstrace[test_step] = test_agent_result.memories[0]
            
            test_obs, test_reward, test_terminated, test_truncated, test_info = test_env.step(test_action)

            if test_terminated or test_truncated:
                if 'episode' in test_info: # 'episode' is a default stats_key for RecordEpisodeStatistics
                    test_episode_stats = test_info['episode']
                    LOG(f'{test_step:05} {test_episode_stats}')
                    
                test_step = 0
                test_hstrace_indices = torch.arange(HP.trxl_memory_length).long().to(test_device)
            else:
                assert not 'episode' in test_info
                test_step += 1

  0%|          | 0/200 [00:00<?, ?it/s]

00045 {'r': 0.0, 'l': 46, 't': 0.32523}
00078 {'r': 0.4, 'l': 78, 't': 0.509118}
00062 {'r': 0.2, 'l': 62, 't': 0.405548}


## Configure

In [21]:
# @launchit.disable
# @launchit.collect_1
HP.trxl_layers_count = 3 # number of transformer layers
HP.trxl_heads_count = 4 # number of heads used in multi-head attention
HP.trxl_d_model = 384 # the dimension of the transformer
HP.trxl_memory_length = 119 # the length of TrXL's sliding memory window
HP.trxl_positional_encoding = "absolute" # positional encoding type of the transformer: "", "absolute", "learned"
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'torch_compile': True,
 'env_id': 'MortarMayhem-Grid-v0',
 'envs_count': 32,
 'trxl_layers_count': 3,
 'trxl_heads_count': 4,
 'trxl_d_model': 384,
 'trxl_memory_length': 119,
 'trxl_positional_encoding': 'absolute',
 'gamma': 0.995,
 'gae_lambda': 0.95,
 'global_steps_count': 1000000,
 'rollout_steps_count': 512,
 'anneal_steps_count': 5120000,
 'minibatches_count': 8,
 'epochs_count': 3,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'init_ent_coef': 0.0001,
 'final_ent_coef': 1e-06,
 'vf_coef': 0.5,
 'max_grad_norm': 0.25,
 'target_kl': None,
 'norm_adv': False,
 'reconstruction_coef': 0.0,
 'capture_video': 'every(1000000)',
 'optimizer': 'AdamW',
 'init_learn_rate': 0.000275,
 'final_learn_rate': 1e-05,
 'workers_count': 1}


## Create

In [22]:
# @launchit.disable_2
ap = Agent.Params(
    d_model=HP.trxl_d_model,
    layers_count=HP.trxl_layers_count,
    heads_count=HP.trxl_heads_count,
    positional_encoding=HP.trxl_positional_encoding,
    observation_space_shape=LS.env_observation_space_shape,
    action_space_shape=LS.env_action_space_shape,
    max_episode_steps_count=LS.env_max_episode_steps_count,
    with_reconstruction_head=False,
)
LS.agent = Agent(ap).to(CONFIG.cuda_device)
LS.agent.share_memory()

if HP.torch_compile:
    LS.agent = torch.compile(LS.agent, fullgraph=True)

# Video

## get_fresh_video_dir_name

In [23]:
def get_fresh_video_dir_name():
    lc = HP.launch_component()
    timestamp = datetime.datetime.now().strftime("%Y.%m.%d-%H:%M:%S") # generate unique dir name in order to shut up RecordVideo from complaining
    return os.path.join(CONFIG.run_path, f'video-{lc.name}-launch{lc.version}-{timestamp}')

## capture_video_of_test_rollout

In [24]:
def capture_video_of_test_rollout(agent, max_steps_count=10_000, video_dir_name=None, random_seed=None):
    video_dir_name = lu.coalesce(video_dir_name, lambda: get_fresh_video_dir_name())
    assert video_dir_name is not None
    env = create_env(HP.env_id, video_dir_name=video_dir_name, is_auto_reset=False, random_seed=random_seed)
    obs, _ = env.reset(seed=random_seed)
    device = next(iter(agent.parameters())).device
    hstrace = torch.zeros((max_steps_count, HP.trxl_layers_count, HP.trxl_d_model), dtype=torch.float32).to(device)
    causal_masks = torch.tril(torch.ones((HP.trxl_memory_length, HP.trxl_memory_length)), diagonal=-1).to(device)
    hstrace_indices = torch.arange(HP.trxl_memory_length).long().to(device)

    with eval_guard(agent):
        with torch.no_grad():
            for step in range(max_steps_count): 
                # hstrace_indices update logic: the rightmost index must be >= step, otherwise - slide window
                if hstrace_indices[-1] < step:
                    hstrace_indices += (test_step - hstrace_indices[-1])
    
                assert hstrace_indices[-1] >= step

                agent_result = agent.get_action_and_value(
                    x=torch.tensor(obs).unsqueeze(0).to(device),
                    memory=hstrace[hstrace_indices].unsqueeze(0),
                    memory_mask=causal_masks[min(step, HP.trxl_memory_length - 1)].unsqueeze(0),
                    memory_indices=hstrace_indices.unsqueeze(0),
                )

                action = agent_result.actions[0].item()
                hstrace[step] = agent_result.memories[0]
                
                obs, reward, terminated, truncated, info = env.step(action)
    
                if terminated or truncated:
                    break

    game_meta = dict(
        reward=env.get_wrapper_attr('episode_returns'),
        frames_count=env.get_wrapper_attr('episode_lengths'),
        steps_count=step + 1,
    )
    
    # force video recording to complete and write video file. For Atari env.close() works well
    # but for pygame based env.close() leads to destruction of shared display which may lead
    # to problems with other envs =)
    # https://share.google/aimode/muVrVExD4U7rWfulg
    env.get_wrapper_attr('stop_recording')()
    del env
    
    video_fnames = list(filter(lambda fn: os.path.isfile(os.path.join(video_dir_name, fn)), os.listdir(video_dir_name)))
    assert len(video_fnames) == 1, len(video_fnames)
    video_fname = os.path.join(video_dir_name, video_fnames[0])
    video_meta = {}
    
    with open(video_fname, 'rb') as f:
        container = av.open(f)
        video_meta['fps'] = float(container.streams.video[0].average_rate)
        video_stream = container.streams.video[0]
        video_meta['duration'] = float(video_stream.duration * video_stream.time_base)
        
    return video_fname, dict(game=game_meta, video=video_meta)

In [25]:
# @launchit.disable
capture_video_of_test_rollout(LS.agent, max_steps_count=LS.env_max_episode_steps_count)

('/home/misha/dev/mine/neurolab/run/17_rl/video-17c_ppo_trxl_memory_mp_01-launch0-2026.05.15-16:20:27/rl-video-episode-0.mp4',
 {'game': {'reward': 0.0, 'frames_count': 46, 'steps_count': 46},
  'video': {'fps': 6.0, 'duration': 7.833333333333333}})

# Utils

## batched_index_select

In [26]:
# See 17c_intuition.ipynb for explanation
def batched_index_select(input, index, dim=1):
    index = index.reshape((*index.shape, 1, 1)) # e.g. [e,s] -> [e,s,1,1]
    index = index.expand((-1, -1, input.shape[2], input.shape[3])) # e.g. [e,s,1,1] -> [e,s,3,384]
    return torch.gather(input, dim, index)

# Worker

Implementation details regarding memory sharing between PyTorch applications: <a href="./dialogs/torch-multiprocessing-shm.ipynb">torch-multiprocessing-shm.ipynb</a>

## WorkerTask

In [27]:
# Exchange data between main and child processes
@dataclass(slots=True)
class WorkerTask:
    task_id: int
    op: str
    params: dict = None

@dataclass(slots=True)
class WorkerTaskResult:
    task_id: int
    payload: object = None

## WorkerCtl

In [28]:
# Master's stuff (main process)
class WorkerCtl:
    task_id = 0
    
    def __init__(self, worker_ind, module, mp_ctx):
        self.task_ctor = getattr(module, 'WorkerTask')
        self.worker_ind = worker_ind
        self.task_queue = mp_ctx.Queue()
        self.task_result_queue = mp_ctx.Queue()
        self.process = mp_ctx.Process(target=getattr(module, 'worker_loop'), args=(worker_ind, self.task_queue, self.task_result_queue))
        self.process.start()
        self.pending_task_ids = deque()

    @staticmethod
    def gen_task_id():
        WorkerCtl.task_id += 1
        return WorkerCtl.task_id

    def healthcheck(self):
        task = self.task_ctor(task_id=self.gen_task_id(), op='HEALTHCHECK')
        self.task_queue.put(task)
        self.task_result_queue.get()
        
    def terminate(self, timeout=None):
        if self.process.is_alive():
            task = self.task_ctor(task_id=self.gen_task_id(), op='TERMINATE')
            try:
                self.task_queue.put(task)
                self.task_result_queue.get(timeout=timeout)
                self.process.join()
            except:
                self.process.terminate()

    def init_agent(self, agent_params, agent_state_dict=None, torch_compile=False, device=None):
        if agent_state_dict is not None:
            for key in agent_state_dict:
                assert agent_state_dict[key].is_shared()
        
        task = self.task_ctor(
            task_id=self.gen_task_id(), 
            op='INIT_AGENT', 
            params=dict(
                agent_params=dataclasses.asdict(agent_params),
                agent_state_dict=agent_state_dict,
                torch_compile=torch_compile,
                device=device,
            ),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def sync_agent(self, agent_state_dict):
        for key in agent_state_dict:
            assert agent_state_dict[key].is_shared()
                
        task = self.task_ctor(
            task_id=self.gen_task_id(),
            op='SYNC_AGENT',
            params=dict(agent_state_dict=agent_state_dict),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def init_rollout(self, env_inds, env_max_episode_steps_count):
        task = self.task_ctor(
            task_id=self.gen_task_id(), 
            op='INIT_ROLLOUT', 
            params=dict(
                env_inds=env_inds,
                env_max_episode_steps_count=env_max_episode_steps_count,
            )
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id
        
    def rollout(self, shared_tensors):
        for k, t in shared_tensors.items():
            assert t.is_shared(), f'Tensor {k} is not shared'
            
        task = self.task_ctor(
            task_id=self.gen_task_id(), 
            op='ROLLOUT', 
            params=dict(
                shared_tensors=shared_tensors,
            ), 
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def capture_video_of_test_rollout(self, max_steps_count, video_dir_name, random_seed=None, forward_data=None):
        task = self.task_ctor(
            task_id=self.gen_task_id(),
            op='CAPTURE_VIDEO',
            params=dict(
                max_steps_count=max_steps_count, 
                video_dir_name=video_dir_name,
                random_seed=random_seed,
                forward_data=forward_data,
            ),
        )
        self.task_queue.put(task)
        self.pending_task_ids.append(task.task_id)
        return task.task_id

    def is_busy(self):
        return len(self.pending_task_ids) > 0
    
    def get_task_result(self):
        assert len(self.pending_task_ids) > 0
        result = self.task_result_queue.get()
        assert result.task_id == self.pending_task_ids.popleft()
        return result

    def peek_task_result(self):
        if not self.pending_task_ids:
            return None
        
        try:
            result = self.task_result_queue.get(block=False)
            assert result.task_id == self.pending_task_ids.popleft()
            return result
        except queue.Empty:
            return None # task is not completed yet

    def drain_task_results(self):
        results = []

        while self.pending_task_ids:
            task_id = self.pending_task_ids.popleft()
            task_result = self.task_result_queue.get()
            assert task_result.task_id == task_id
            results.append(task_result)

        return results

## create_shared_tensors

In [29]:
def create_shared_tensors(
    envs_count, 
    env_observation_space_shape, 
    env_action_space_shape,
    env_max_episode_steps_count, 
    rollout_steps_count, 
    layers_count, 
    d_model, 
    memory_length, 
):
    shared_tensors = dict(
        obs=torch.zeros((rollout_steps_count, envs_count, *env_observation_space_shape)).to(CONFIG.cuda_device),
        rewards=torch.zeros((rollout_steps_count, envs_count)),
        dones=torch.zeros((rollout_steps_count, envs_count)),
        actions=torch.zeros((rollout_steps_count, envs_count, len(env_action_space_shape)), dtype=torch.long).to(CONFIG.cuda_device),
        action_log_probs=torch.zeros((rollout_steps_count, envs_count, len(env_action_space_shape))).to(CONFIG.cuda_device),
        values=torch.zeros((rollout_steps_count, envs_count)).to(CONFIG.cuda_device),
        advantages=torch.zeros((rollout_steps_count, envs_count)).to(CONFIG.cuda_device),
        returns=torch.zeros((rollout_steps_count, envs_count)).to(CONFIG.cuda_device),
        training_causal_masks=torch.zeros((rollout_steps_count, envs_count, memory_length), dtype=torch.bool).to(CONFIG.cuda_device),
        training_hstrace_indices=torch.zeros((rollout_steps_count, envs_count, memory_length), dtype=torch.long).to(CONFIG.cuda_device),
    )
    
    for t in shared_tensors.values():
        t.share_memory_()

    return shared_tensors

## worker_loop

In [30]:
# Executed in child process
def worker_loop(worker_ind, task_queue, task_result_queue):
    @dataclass(slots=True)
    class WorkerState:
        device: str = None
        agent: object = None
        is_attached_agent: bool = False
        env_inds: object = None
        my_env_inds: object = None
        envs: object = None
        next_obs: object = None
        next_dones: object = None
        episode_steps: object = None
        episode_hstraces: object = None
        episode_causal_masks: object = None
        episode_hstrace_indices: object = None
    
    CONFIG = create_config()
    LOG = Logging.get()
    LOG.app_name = CONFIG.self_name
    LOG.enable('syslog', True)
    LOG.enable('stdout', False)
    worker_random_seed = HP.random_seed + worker_ind
    
    with LOG.auto_prefix('WRK', worker_ind, 'SEED', worker_random_seed):
        LOG(f'CONFIG={CONFIG._asdict()}')
        
        au.init()
        random.seed(worker_random_seed)
        torch.manual_seed(worker_random_seed)
        RNG = np.random.default_rng(worker_random_seed)
        LOG(f'{worker_random_seed=}')
        
        torch.backends.cudnn.deterministic = HP.torch_deterministic
        LOG(f'{torch.backends.cudnn.deterministic=}')

        WS = WorkerState()
        LOG('Worker is ready')
        
        task_wait_timeout = 60
        is_running = True
    
        while is_running:
            try:
                # task is expected to be an instanace of WorkerTask class
                task = task_queue.get(block=True, timeout=task_wait_timeout)
            except queue.Empty:
                LOG(f'Didn\'t get any tasks within {task_wait_timeout} seconds, waiting again')
                continue

            with LOG.auto_prefix('TASK', task.task_id):
                LOG(f'Got task #{task.task_id} {task.op}')
                task_result = WorkerTaskResult(task_id=task.task_id)
                
                match task.op:
                    case 'HEALTHCHECK':
                        pass
                    case 'TERMINATE':
                        is_running = False
                    case 'INIT_AGENT':
                        ap = Agent.Params(**task.params['agent_params'])
                        WS.device = lu.coalesce(task.params.get('device'), CONFIG.cuda_device)
                        WS.agent = Agent(ap).to(WS.device)
                        
                        if task.params['torch_compile']:
                            WS.agent = torch.compile(WS.agent, fullgraph=True)
                            LOG('Agent compiled')
                            
                        WS.is_attached_agent = task.params['agent_state_dict'] is not None
                        
                        if WS.is_attached_agent:
                            # Storage for weights of attached agent is pointed to agent's weights in main process
                            state_dict = task.params['agent_state_dict']
                            
                            with torch.no_grad():
                                for name, param in WS.agent.named_parameters():
                                    assert state_dict[name].is_shared()
                                    param.data = state_dict[name]

                        LOG(f'{WS.is_attached_agent=}')
                    case 'SYNC_AGENT':
                        assert WS.agent is not None
                        assert not WS.is_attached_agent
                        
                        state_dict = task.params['agent_state_dict']
                        
                        with torch.no_grad():
                            # Fast GPU-TO-GPU sync
                            for name, param in WS.agent.named_parameters():
                                assert state_dict[name].is_shared()
                                param.copy_(state_dict[name])
                    case 'INIT_ROLLOUT':
                        assert WS.agent is not None
                        
                        WS.env_inds = task.params['env_inds']
                        envs_count = len(WS.env_inds)
                        WS.my_env_inds = torch.arange(envs_count)
                        env_max_episode_steps_count = task.params['env_max_episode_steps_count']

                        def create_episode_hstrace_indices(max_episode_steps_count, memory_length):
                            repetitions = torch.repeat_interleave(
                                torch.arange(0, memory_length).unsqueeze(0), 
                                memory_length - 1, 
                                dim=0
                            ).long()
                            episode_hstrace_indices = torch.stack(
                                [torch.arange(i, i + memory_length) for i in range(max_episode_steps_count - memory_length + 1)]
                            ).long()
                            return torch.cat((repetitions, episode_hstrace_indices))

                        
                        # Current episode tracker and hstrace for each env
                        WS.episode_steps = torch.zeros(envs_count, dtype=torch.long)
                        WS.episode_hstraces = torch.zeros((envs_count, env_max_episode_steps_count, WS.agent.params.layers_count, WS.agent.params.d_model), dtype=torch.float32).to(WS.device)
                        WS.episode_causal_masks = torch.tril(torch.ones((HP.trxl_memory_length, HP.trxl_memory_length)), diagonal=-1).to(WS.device)
                        WS.episode_hstrace_indices = create_episode_hstrace_indices(env_max_episode_steps_count, HP.trxl_memory_length).to(WS.device)
                        
                        env_factory = lambda random_seed: lambda: create_env(HP.env_id, random_seed=random_seed)
                        WS.envs = gym.vector.SyncVectorEnv(
                            [env_factory(HP.random_seed + int(env_ind)) for env_ind in WS.env_inds],
                            autoreset_mode=gym.vector.vector_env.AutoresetMode.NEXT_STEP, # https://farama.org/Vector-Autoreset-Mode
                        )
                        LOG(f'Envs created, env_inds=[{','.join(map(lambda i: str(i.item()), WS.env_inds))}]')

                        WS.next_obs, _ = WS.envs.reset(seed=worker_random_seed)
                        WS.next_obs = torch.Tensor(WS.next_obs).to(WS.device)
                        WS.next_dones = torch.zeros(len(WS.env_inds))
                        assert len(WS.next_obs) == len(WS.next_dones), (WS.next_obs.shape, WS.next_dones.shape)
                        LOG('Envs reset')
                    case 'ROLLOUT':
                        assert WS.agent is not None
                        assert WS.envs is not None
                        
                        shared_tensors = task.params['shared_tensors']
                        obs, rewards, dones, actions, action_log_probs, values, advantages, returns = (
                            shared_tensors['obs'], 
                            shared_tensors['rewards'], 
                            shared_tensors['dones'], 
                            shared_tensors['actions'],
                            shared_tensors['action_log_probs'],
                            shared_tensors['values'],
                            shared_tensors['advantages'],
                            shared_tensors['returns'],
                        )
                        training_causal_masks, training_hstrace_indices = (
                            shared_tensors['training_causal_masks'],
                            shared_tensors['training_hstrace_indices'],
                        )
                        rollout_steps_count = len(obs)
                        # ROLLOUT AND COLLECT hstraces for subsequent training
                        my_episode_stats = []
                        my_training_hstraces = defaultdict(dict) # env_ind -> { ending_step (inclusive) -> hstrace }

                        with torch.no_grad():
                            for step in range(rollout_steps_count): 
                                obs[step,WS.env_inds] = WS.next_obs
                                dones[step,WS.env_inds] = WS.next_dones
                                current_hstrace_indices = WS.episode_hstrace_indices[WS.episode_steps] # [env_inds, HP.trxl_memory_length]
                                current_causal_masks = WS.episode_causal_masks[torch.clip(WS.episode_steps, 0, HP.trxl_memory_length - 1)]  # [env_inds, HP.trxl_memory_length]
                                training_causal_masks[step,WS.env_inds] = current_causal_masks.bool()
                                training_hstrace_indices[step,WS.env_inds] = current_hstrace_indices
                                hidden_states = batched_index_select(WS.episode_hstraces, current_hstrace_indices) # See 17c_intuition.ipynb for explanation
                                
                                agent_result = WS.agent.get_action_and_value(
                                    x=WS.next_obs, 
                                    memory=hidden_states, 
                                    memory_mask=current_causal_masks, 
                                    memory_indices=current_hstrace_indices,
                                )

                                # See 17c_intuition for why [WS.my_env_inds,WS.episode_steps] indexing should be used instead of simple [:,WS.episode_steps]
                                WS.episode_hstraces[WS.my_env_inds,WS.episode_steps] = agent_result.memories
                                actions[step,WS.env_inds] = agent_result.actions
                                action_log_probs[step,WS.env_inds] = agent_result.action_log_probs
                                values[step,WS.env_inds] = agent_result.values

                                WS.next_obs, _rewards, terminations, truncations, infos = WS.envs.step(agent_result.actions.cpu().numpy())
                                WS.next_obs = torch.Tensor(WS.next_obs).to(WS.device, non_blocking=True)
                                WS.next_dones = torch.Tensor(np.logical_or(terminations, truncations))
                                rewards[step,WS.env_inds] = torch.tensor(_rewards.astype(np.float32)).view(-1)

                                for my_env_ind, done in enumerate(WS.next_dones):
                                    if not done:
                                        # Game continues, just advance episode_step further
                                        WS.episode_steps[my_env_ind] += 1
                                    else:
                                        # Game over, save episodic_hstrace and reset it for new episoode
                                        WS.episode_steps[my_env_ind] = 0
                                        
                                        env_ind = WS.env_inds[my_env_ind].item()
                                        
                                        # clone episode_hstrace so the latter is retained in memory and is available to master process
                                        my_training_hstraces[env_ind][step] = WS.episode_hstraces[my_env_ind].clone()
                                        assert my_training_hstraces[env_ind][step].is_shared()
                                        
                                        WS.episode_hstraces[my_env_ind].zero_()
                        
                                if 'episode' in infos: # 'episode' is a default stats_key for RecordEpisodeStatistics
                                    episode_stats = infos['episode']
                                    assert np.all(np.argwhere(episode_stats['_l']) == np.argwhere(episode_stats['_r']))
                                    
                                    for my_env_ind in np.argwhere(episode_stats['_r']):
                                        my_episode_stats_item = dict(
                                            env_ind=WS.env_inds[my_env_ind].item(),
                                            l=episode_stats['l'][my_env_ind].item(), 
                                            r=episode_stats['r'][my_env_ind].item()
                                        )
                                        my_episode_stats.append(my_episode_stats_item)

                        shared_tensors['next_obs'] = WS.next_obs 
                        shared_tensors['next_dones'] = WS.next_dones
                        
                        for my_env_ind, env_ind in enumerate(WS.env_inds):
                            env_ind = env_ind.item()
                            assert step == rollout_steps_count - 1
                            my_training_hstraces[env_ind][step] = WS.episode_hstraces[my_env_ind].clone() 
                            assert my_training_hstraces[env_ind][step].is_shared()

                        # ADVANTAGES
                        with torch.no_grad():
                            indices = WS.episode_hstrace_indices[WS.episode_steps]
                            hidden_states = batched_index_select(WS.episode_hstraces, indices) 
                            next_values = WS.agent.get_value(
                                x=WS.next_obs,
                                memory=hidden_states,
                                memory_mask=WS.episode_causal_masks[torch.clip(WS.episode_steps, 0, HP.trxl_memory_length - 1)],
                                memory_indices=indices,
                            ).cpu()
                            my_rewards = rewards[:,WS.env_inds]
                            my_values = values[:,WS.env_inds].cpu()
                            my_advantages = advantages[:,WS.env_inds].cpu()
                            lastgaelam = 0
                            
                            for t in reversed(range(rollout_steps_count)):
                                if t == rollout_steps_count - 1:
                                    nextnonterminal = 1.0 - WS.next_dones
                                    nextvalues = next_values
                                else:
                                    nextnonterminal = 1.0 - dones[t+1,WS.env_inds]
                                    nextvalues = my_values[t+1]
                                
                                delta = my_rewards[t] + HP.gamma * nextvalues * nextnonterminal - my_values[t]
                                lastgaelam = delta + HP.gamma * HP.gae_lambda * nextnonterminal * lastgaelam
                                my_advantages[t] = lastgaelam

                            advantages[:,WS.env_inds] = my_advantages.to(WS.device)
                            returns[:,WS.env_inds] = advantages[:,WS.env_inds] + values[:,WS.env_inds]

                        task_result.payload = dict(
                            training_hstraces=my_training_hstraces,
                            episode_stats=my_episode_stats,
                        )
                        LOG(f'Done rollout for {rollout_steps_count} steps')
                    case 'CAPTURE_VIDEO':
                        assert WS.agent is not None
                        
                        with torch.no_grad():
                            video_fname, video_meta = capture_video_of_test_rollout(
                                WS.agent, 
                                max_steps_count=task.params['max_steps_count'], 
                                video_dir_name=task.params['video_dir_name'],
                                random_seed=task.params['random_seed'],
                            )
                            task_result.payload = (video_fname, video_meta, task.params['forward_data'])
                    case _:
                        LOG(f'Unknown {task.op=}, ignoring')
        
                task_result_queue.put(task_result)
                LOG('Task complete')
        
        LOG('Worker is going down')

## get_worker_factory

In [31]:
def get_worker_factory():
    with LOG.auto_log_level(logging.INFO):
        expandvars = dict(
            PROJECT_ROOT_PATH=CONFIG.project_root_path,
            MODEL_NAME=CONFIG.self_name,
            MODEL_VERSION=HP.launch_component().version,
            LAUNCH_GOAL=LaunchGoal.WORKER.value,
        )
        module_fname = launchit.launchit(
            CONFIG.self_fname, 
            expandvars=expandvars, 
            make_py_file=True,
            dir_name=CONFIG.run_path,
            disable_inds=[2]
        )
        LOG.info(f'Created "{module_fname}"')
        
    module_dir_name = os.path.dirname(module_fname)
    module_name = os.path.splitext(os.path.basename(module_fname))[0]
    sys.path.append(module_dir_name)
    module = __import__(module_name)

    def factory(worker_ind):
        return WorkerCtl(worker_ind, module, LS.mp_ctx)

    return factory

## Test

### rollout

In [32]:
# @launchit.disable
test_envs_count = 32
test_workers_count = 2

test_wf = get_worker_factory()
test_workers = [test_wf(i) for i in range(test_workers_count)]

try:
    test_envs_count_per_worker = test_envs_count // test_workers_count
    test_env_inds = torch.arange(test_envs_count)
    
    for w_ind, w in enumerate(test_workers): 
        w.init_agent(agent_params=LS.agent.params, agent_state_dict=LS.agent.state_dict(), torch_compile=HP.torch_compile)
        test_lo_env_inds = w_ind * test_envs_count_per_worker
        test_hi_env_inds = lu.when((w_ind + 1) < test_workers_count, (w_ind + 1) * test_envs_count_per_worker, None)
        w.init_rollout(test_env_inds[test_lo_env_inds:test_hi_env_inds], LS.env_max_episode_steps_count)
        w.drain_task_results()
    
    test_rollout_steps_count = 100
    test_global_steps_count = test_rollout_steps_count * test_envs_count * 10

    test_shared_tensors = create_shared_tensors(
        envs_count=test_envs_count,
        env_observation_space_shape=LS.env_observation_space_shape, 
        env_action_space_shape=LS.env_action_space_shape,
        env_max_episode_steps_count=LS.env_max_episode_steps_count, 
        rollout_steps_count=test_rollout_steps_count, 
        layers_count=HP.trxl_layers_count, 
        d_model=HP.trxl_d_model, 
        memory_length=HP.trxl_memory_length, 
    )

    test_step = 0
    test_episode_stats_mafs = dict(l=RecursiveMovingAverageFilter(100), r=RecursiveMovingAverageFilter(100))
    
    with tqdm(total=test_global_steps_count) as pbar:
        while test_step < test_global_steps_count:
            for w in test_workers:
                w.rollout(test_shared_tensors)

            for w in test_workers:
                tr = w.get_task_result().payload

                for episode_stats_item in tr['episode_stats']:
                    test_episode_stats_mafs['l'](episode_stats_item['l'])
                    test_episode_stats_mafs['r'](episode_stats_item['r'])

                test_training_hstraces = tr.pop('training_hstraces')
                # Pretend we work here with test_training_hstraces.
                # Need to delete it to avoid warning/error "Producer process has been terminated before all shared CUDA tensors released"
                # One may also call torch.cuda.ipc_collect() to ensure shared training_hstraces is released
                del test_training_hstraces

            step_inc = test_rollout_steps_count * test_envs_count
            test_step += step_inc
            pbar.update(step_inc)

    print(test_episode_stats_mafs)
finally:
    for w in test_workers: 
        w.terminate(timeout=3)

### capture_video_of_test_rollout

In [33]:
# @launchit.disable
wf = get_worker_factory()
test_worker = wf(0)

try:
    test_worker.init_agent(LS.agent.params)
    test_worker.get_task_result()
    
    video_dir_name = get_fresh_video_dir_name()
    # _orig_mod may be introduced when using torch.compile()
    state_dict = lu.when(hasattr(LS.agent, '_orig_mod'), lambda: LS.agent._orig_mod.state_dict(), lambda: LS.agent.state_dict())
    tid = test_worker.sync_agent(state_dict)
    test_worker.capture_video_of_test_rollout(max_steps_count=2000, video_dir_name=video_dir_name)
    tr = test_worker.get_task_result() # wait for weights are synced is done
    assert tr.task_id == tid
    result = test_worker.drain_task_results()[-1]
    print(result)
finally:
    test_worker.terminate()

## Configure

In [34]:
# @launchit.disable
# @launchit.collect_1
HP.workers_count = 2
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'torch_compile': True,
 'env_id': 'MortarMayhem-Grid-v0',
 'envs_count': 32,
 'trxl_layers_count': 3,
 'trxl_heads_count': 4,
 'trxl_d_model': 384,
 'trxl_memory_length': 119,
 'trxl_positional_encoding': 'absolute',
 'gamma': 0.995,
 'gae_lambda': 0.95,
 'global_steps_count': 1000000,
 'rollout_steps_count': 512,
 'anneal_steps_count': 5120000,
 'minibatches_count': 8,
 'epochs_count': 3,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'init_ent_coef': 0.0001,
 'final_ent_coef': 1e-06,
 'vf_coef': 0.5,
 'max_grad_norm': 0.25,
 'target_kl': None,
 'norm_adv': False,
 'reconstruction_coef': 0.0,
 'capture_video': 'every(1000000)',
 'optimizer': 'AdamW',
 'init_learn_rate': 0.000275,
 'final_learn_rate': 1e-05,
 'workers_count': 2}


## Create

In [35]:
# @launchit.disable_2
wf = get_worker_factory()
LS.workers = [wf(i) for i in range(HP.workers_count)]

envs_count_per_worker = HP.envs_count // HP.workers_count
env_inds = torch.arange(HP.envs_count)

# Init in serial, shows to be much faster than in parallel (GPU contention issues?)
for w_ind, w in enumerate(LS.workers): 
    w.init_agent(agent_params=LS.agent.params, agent_state_dict=LS.agent.state_dict(), torch_compile=HP.torch_compile)
    lo_env_inds = w_ind * envs_count_per_worker
    hi_env_inds = lu.when((w_ind + 1) < HP.workers_count, (w_ind + 1) * envs_count_per_worker, None)
    w.init_rollout(env_inds[lo_env_inds:hi_env_inds], LS.env_max_episode_steps_count)
    w.drain_task_results()

LS.capture_video_worker = wf(0)
LS.capture_video_worker.init_agent(agent_params=LS.agent.params)
LS.capture_video_worker.get_task_result()

Created "/home/misha/dev/mine/neurolab/run/17_rl/17c_ppo_trxl_memory_mp_01-launch98.py"


/home/misha/anaconda3/envs/mine/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/home/misha/anaconda3/envs/mine/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/home/misha/anaconda3/envs/mine/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is sla

WorkerTaskResult(task_id=5, payload=None)

# TRAIN

## CaptureVideoManager

In [36]:
class CaptureVideoManager:
    def __init__(self, capture_video_policy):
        ump = hp_parse_universal_module(capture_video_policy)
        self.should_capture_video = self.create_should_capture_video(ump.module_name, *ump.args, **ump.kwargs)
    
    def schedule_capture_video(self, global_step):
        # _orig_mod may be introduced when using torch.compile()
        state_dict = lu.when(hasattr(LS.agent, '_orig_mod'), lambda: LS.agent._orig_mod.state_dict(), lambda: LS.agent.state_dict())
        task_id = LS.capture_video_worker.sync_agent(state_dict)
        LS.capture_video_worker.capture_video_of_test_rollout(
            max_steps_count=LS.env_max_episode_steps_count, 
            video_dir_name=get_fresh_video_dir_name(),
            random_seed=HP.random_seed,
            forward_data=dict(global_step=global_step)
        )
        task_result = LS.capture_video_worker.get_task_result() # wait until sync weights is finished so we capture video on agent with weights as LS.agent
        assert task_id == task_result.task_id

    def upload_captured_video(self, is_drain=False):
        if is_drain:
            trs = LS.capture_video_worker.drain_task_results()
        else:
            trs = [LS.capture_video_worker.peek_task_result()]

        for tr in filter(lambda tr: tr is not None, trs):
            video_fname, video_meta, forward_data = tr.payload
            _, video_fname_ext = os.path.splitext(video_fname)
            ts = datetime.datetime.now().strftime('%Y.%m.%d-%H:%M:%S')
            remote_video_fname = f'{ts}-{forward_data['global_step']:09}.{video_fname_ext.lstrip('.')}'
            summary_writer.add_file(video_fname, remote_video_fname)
            summary_writer.add_file(io.StringIO(json.dumps(video_meta)), remote_video_fname + '.meta')
            ref_text = f'<a href="http://tensorboard-videos:6007/{summary_writer.log_dir}/{remote_video_fname}" target="_blank">{remote_video_fname}</a>'
            summary_writer.add_text('videos', ref_text, forward_data['global_step'])

    @staticmethod
    def create_should_capture_video(policy_name, period):
        if policy_name == 'every':
            last_count = 0
            
            def every_thunk(count):
                nonlocal last_count
                elapsed = count - last_count

                if count == 0 or elapsed >= period:
                    last_count = count
                    return True

                return False

            return every_thunk
        
        assert False, f'Unsupported {policy_name=}'

## Configure

In [37]:
# @launchit.disable
# @launchit.collect

HP.gamma = 0.995 # return discount factor gamma
HP.gae_lambda = 0.95 # lambda for the general advantage estimation

# Training procedure params (PPO related) 
HP.global_steps_count = 200_000 # total number of steps 
HP.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
HP.anneal_steps_count = 32 * 512 * 10_000 # anneal steps count for learn rate and entropy coeff
HP.minibatches_count = 8
HP.epochs_count = 3 
HP.clip_coef = 0.1 # the surrogate clipping coefficient
HP.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
HP.init_ent_coef = 0.0001 # initial coefficient of the entropy
HP.final_ent_coef = 0.000001 # final coefficient of the entropy
HP.vf_coef = 0.5 # coefficient of the value function
HP.max_grad_norm = 0.25 # the maximum norm for the gradient clipping
HP.target_kl = None # e target KL divergence threshold
HP.norm_adv = False # Toggles advantages normalization
HP.reconstruction_coef = 0.0 # the coefficient of the observation reconstruction loss, if set to 0.0 the reconstruction loss is not used

# Video params
HP.capture_video = 'every(1000000)' # video capture policy depending on steps

# Optimization params
HP.optimizer = 'AdamW'
HP.init_learn_rate = 2.75e-4
HP.final_learn_rate = 1.0e-5

# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'random_seed': 42,
 'torch_deterministic': True,
 'torch_compile': True,
 'env_id': 'MortarMayhem-Grid-v0',
 'envs_count': 32,
 'trxl_layers_count': 3,
 'trxl_heads_count': 4,
 'trxl_d_model': 384,
 'trxl_memory_length': 119,
 'trxl_positional_encoding': 'absolute',
 'gamma': 0.995,
 'gae_lambda': 0.95,
 'global_steps_count': 200000,
 'rollout_steps_count': 512,
 'anneal_steps_count': 163840000,
 'minibatches_count': 8,
 'epochs_count': 3,
 'clip_coef': 0.1,
 'clip_vloss': True,
 'init_ent_coef': 0.0001,
 'final_ent_coef': 1e-06,
 'vf_coef': 0.5,
 'max_grad_norm': 0.25,
 'target_kl': None,
 'norm_adv': False,
 'reconstruction_coef': 0.0,
 'capture_video': 'every(1000000)',
 'optimizer': 'AdamW',
 'init_learn_rate': 0.000275,
 'final_learn_rate': 1e-05,
 'workers_count': 2}


## Create

In [38]:
# @launchit.disable_2
ump = hp_parse_universal_module(HP.optimizer)
assert not ump.args
optimizer = getattr(torch.optim, ump.module_name)(LS.agent.parameters(), lr=HP.init_learn_rate, **ump.kwargs)

learn_rate_anneal = get_linear_anneal(HP.init_learn_rate, HP.final_learn_rate, HP.anneal_steps_count)
ent_coef_anneal = get_linear_anneal(HP.init_ent_coef, HP.final_ent_coef, HP.anneal_steps_count)

capture_video_manager = CaptureVideoManager(HP.capture_video)

In [39]:
# @launchit.disable_2
episode_stats_mafs = dict(l=RecursiveMovingAverageFilter(max_n=100), r=RecursiveMovingAverageFilter(max_n=100))

shared_tensors = create_shared_tensors(
    envs_count=HP.envs_count,
    env_observation_space_shape=LS.env_observation_space_shape, 
    env_action_space_shape=LS.env_action_space_shape,
    env_max_episode_steps_count=LS.env_max_episode_steps_count, 
    rollout_steps_count=HP.rollout_steps_count, 
    layers_count=HP.trxl_layers_count, 
    d_model=HP.trxl_d_model, 
    memory_length=HP.trxl_memory_length, 
)

for name, v in shared_tensors.items():
    assert v.is_shared(), f'{name} is not shared'
    LOG(f'{name:>25}: {str(v.device):>6}, {str(v.dtype):>15}, {v.shape}')

training_hstrace_map = torch.zeros((HP.rollout_steps_count, HP.envs_count), dtype=torch.long)

                      obs: cuda:0,   torch.float32, torch.Size([512, 32, 84, 84, 3])
                  rewards:    cpu,   torch.float32, torch.Size([512, 32])
                    dones:    cpu,   torch.float32, torch.Size([512, 32])
                  actions: cuda:0,     torch.int64, torch.Size([512, 32, 1])
         action_log_probs: cuda:0,   torch.float32, torch.Size([512, 32, 1])
                   values: cuda:0,   torch.float32, torch.Size([512, 32])
               advantages: cuda:0,   torch.float32, torch.Size([512, 32])
                  returns: cuda:0,   torch.float32, torch.Size([512, 32])
    training_causal_masks: cuda:0,      torch.bool, torch.Size([512, 32, 119])
 training_hstrace_indices: cuda:0,     torch.int64, torch.Size([512, 32, 119])


## Train

In [40]:
# @launchit.disable_2
batch_size = int(HP.envs_count * HP.rollout_steps_count)
minibatch_size = batch_size // HP.minibatches_count
pbar = tqdm(total=HP.global_steps_count)
global_step = 0
start_time = time.time()

while global_step < HP.global_steps_count:
    # ANNEAL STUFF
    learn_rate = learn_rate_anneal(global_step)
    for param_group in optimizer.param_groups: param_group["lr"] = learn_rate
    ent_coef = ent_coef_anneal(global_step)

    # ROLLOUT
    for w in LS.workers:
        w.rollout(shared_tensors)

    # Collect results and construct training_hstraces. 
    # Might include several hstraces per environment in case of game overs. Initially it's just current episode_hstraces
    training_hstraces = []
    training_hstrace_map.fill_(-1)
        
    for w in LS.workers:
        tr = w.get_task_result().payload

        for episode_stats_item in tr['episode_stats']:
            episode_stats_mafs['l'](episode_stats_item['l'])
            episode_stats_mafs['r'](episode_stats_item['r'])

        # use pop() to avoid dangling shared tensors.
        # Reason: training_hstraces are created in (and hence are owned by) worker process which might finish before main process
        for env_ind, env_training_hstraces in tr.pop('training_hstraces').items():
            start_step = 0
            
            for ending_step, env_training_hstrace in env_training_hstraces.items():
                assert 0 <= start_step < HP.rollout_steps_count
                assert 0 <= ending_step < HP.rollout_steps_count
                assert start_step <= ending_step
                training_hstraces.append(env_training_hstrace)
                training_hstrace_map[start_step:ending_step+1,env_ind] = len(training_hstraces) - 1
                start_step = ending_step + 1

    assert torch.all(training_hstrace_map >= 0)

    training_hstraces = torch.stack(training_hstraces, dim=0).to(CONFIG.cuda_device)

    # TRAINING
    b_obs = shared_tensors['obs'].reshape(-1, *shared_tensors['obs'].shape[2:])
    b_actions = shared_tensors['actions'].reshape(-1, *shared_tensors['actions'].shape[2:])
    b_action_log_probs = shared_tensors['action_log_probs'].reshape(-1, *shared_tensors['action_log_probs'].shape[2:])
    b_values = shared_tensors['values'].reshape(-1)
    b_advantages = shared_tensors['advantages'].reshape(-1)
    b_returns = shared_tensors['returns'].reshape(-1)
    b_training_hstrace_map = training_hstrace_map.reshape(-1)
    b_training_hstrace_indices = shared_tensors['training_hstrace_indices'].reshape(-1, *shared_tensors['training_hstrace_indices'].shape[2:])
    b_training_causal_masks = shared_tensors['training_causal_masks'].reshape(-1, *shared_tensors['training_causal_masks'].shape[2:])
    
    # Remove unnecessary padding from TrXL memory, if there are too few steps
    # kms@ review. Is it really needed?
    actual_max_episode_steps = (shared_tensors['training_hstrace_indices'] * shared_tensors['training_causal_masks']).max().item() + 1
    
    if actual_max_episode_steps < HP.trxl_memory_length:
        b_training_hstrace_indices = b_training_hstrace_indices[:, :actual_max_episode_steps]
        b_training_causal_masks = b_training_causal_masks[:, :actual_max_episode_steps]
        training_hstraces = training_hstraces[:, :actual_max_episode_steps]

    # Optimizing the policy and value network
    clipfracs = []
    
    for epoch in range(HP.epochs_count):
        b_inds = torch.randperm(batch_size)
        
        for start in range(0, batch_size, minibatch_size):
            mb_inds = b_inds[start:start + minibatch_size]
            mb_hstraces = training_hstraces[b_training_hstrace_map[mb_inds]]
            mb_hidden_states = batched_index_select(mb_hstraces, b_training_hstrace_indices[mb_inds])

            _, new_action_log_probs, new_entropies, new_values, _ = LS.agent.get_action_and_value(
                x=b_obs[mb_inds], 
                memory=mb_hidden_states, 
                memory_mask=b_training_causal_masks[mb_inds], 
                memory_indices=b_training_hstrace_indices[mb_inds], 
                action=b_actions[mb_inds],
            )

            # Policy loss
            mb_advantages = b_advantages[mb_inds]
            
            if HP.norm_adv:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)
            
            mb_advantages = mb_advantages.unsqueeze(1).repeat(
                1, len(LS.env_action_space_shape)
            )  # Repeat is necessary for multi-discrete action spaces
            
            logratio = new_action_log_probs - b_action_log_probs[mb_inds]
            ratio = torch.exp(logratio)
            pgloss1 = -mb_advantages * ratio
            pgloss2 = -mb_advantages * torch.clamp(ratio, 1.0 - HP.clip_coef, 1.0 + HP.clip_coef)
            pg_loss = torch.max(pgloss1, pgloss2).mean()

            # Value loss
            v_loss_unclipped = (new_values - b_returns[mb_inds]) ** 2
            
            if HP.clip_vloss:
                v_loss_clipped = b_values[mb_inds] + (new_values - b_values[mb_inds]).clamp(min=-HP.clip_coef, max=HP.clip_coef)
                v_loss = torch.max(v_loss_unclipped, (v_loss_clipped - b_returns[mb_inds]) ** 2).mean()
            else:
                v_loss = v_loss_unclipped.mean()

            # Entropy loss
            entropy_loss = new_entropies.mean()

            # Combined losses
            loss = pg_loss - ent_coef * entropy_loss + v_loss * HP.vf_coef

            # Add reconstruction loss if used
            if HP.reconstruction_coef > 0.0:
                r_loss = F.binary_cross_entropy(LS.agent.reconstruct_observation(), b_obs[mb_inds] / 255.0)
                loss += HP.reconstruction_coef * r_loss
            else:
                r_loss = torch.tensor(0) # create dummy one since r_loss is reported to tensorboard

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(LS.agent.parameters(), max_norm=HP.max_grad_norm)
            optimizer.step()

            with torch.no_grad():
                # calculate approx_kl http://joschu.net/blog/kl-approx.html
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs += [((ratio - 1.0).abs() > HP.clip_coef).float().mean().item()]

        if HP.target_kl is not None and approx_kl > HP.target_kl:
            break

    y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

    # REPORT
    summary_writer.add_scalar('charts/sps', int(global_step / (time.time() - start_time)), global_step, is_batched=True)
    summary_writer.add_scalar('charts/learning_rate', learn_rate, global_step, is_batched=True)
    summary_writer.add_scalar('charts/entropy_coefficient', ent_coef, global_step, is_batched=True)
    
    summary_writer.add_scalar('losses/policy_loss', pg_loss, global_step, is_batched=True)
    summary_writer.add_scalar('losses/value_loss', v_loss, global_step, is_batched=True)
    summary_writer.add_scalar('losses/loss', loss, global_step, is_batched=True)
    summary_writer.add_scalar('losses/entropy', entropy_loss, global_step, is_batched=True)
    summary_writer.add_scalar('losses/reconstruction_loss', r_loss, global_step, is_batched=True)
    summary_writer.add_scalar('losses/approx_kl', approx_kl, global_step, is_batched=True)
    summary_writer.add_scalar('losses/clipfrac', np.mean(clipfracs), global_step, is_batched=True)
    summary_writer.add_scalar('losses/explained_variance', explained_var, global_step, is_batched=True)
    
    summary_writer.add_scalar('episode/l_mean', episode_stats_mafs['l'].v, global_step, is_batched=True)
    summary_writer.add_scalar('episode/r_mean', episode_stats_mafs['r'].v, global_step, is_batched=True)
    summary_writer.add_scalar('episode/value_mean', shared_tensors['values'].mean(), global_step, is_batched=True)
    summary_writer.add_scalar('episode/advantage_mean', shared_tensors['advantages'].mean(), global_step, is_batched=True)

    # VIDEO
    if capture_video_manager.should_capture_video(global_step) or (global_step + batch_size >= HP.global_steps_count):
        capture_video_manager.schedule_capture_video(global_step)

    capture_video_manager.upload_captured_video()
    summary_writer.flush()
    
    pbar.update(min(batch_size, HP.global_steps_count - global_step)) # min is used to not overflow progress bar at the end (global_step could get > HP.global_step_size)
    global_step += batch_size

capture_video_manager.upload_captured_video(is_drain=True)
summary_writer.flush()
pbar.close()

  0%|          | 0/200000 [00:00<?, ?it/s]

{'rollout': RecursiveAverageFilter(v=5.852187376755935, n=13),
 'training_hstraces': RecursiveAverageFilter(v=0.021475828610933743, n=13),
 'training': RecursiveAverageFilter(v=1.559003261419443, n=13)}

In [ ]:
# for epoch in tqdm(range(100)):
#     b_inds = torch.randperm(batch_size)
        
#     for start in range(0, batch_size, minibatch_size):
#         mb_inds = b_inds[start:start + minibatch_size]
#         mb_hstraces = training_hstraces[b_training_hstrace_map[mb_inds]]
#         mb_hidden_states = batched_index_select(mb_hstraces, b_training_hstrace_indices[mb_inds])
#         b_obs[mb_inds]
#         mb_hidden_states
#         b_training_causal_masks[mb_inds]
#         b_training_hstrace_indices[mb_inds]
#         b_actions[mb_inds]
#         b_actions[mb_inds]
#         b_advantages[mb_inds]
#         b_action_log_probs[mb_inds]
#         b_returns[mb_inds]
#         b_values[mb_inds]
#         b_values[mb_inds]
#         b_returns[mb_inds]
#         continue

## Save

In [ ]:
# @launchit.disable_2
artifact_registry = LS.new_artifact_registry()
lc = HP.launch_component()

with io.BytesIO() as b:
    torch.save(LS.agent.state_dict(), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='agent', replace=True)

with io.StringIO() as b:
    json.dump(dataclasses.asdict(LS.agent.params), b)
    artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='agent_params', replace=True)

# LaunchIt!

## TRAIN

In [10]:
# @launchit.disable
launchit_t0 = time.time()

In [11]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    artifact_registry = LS.new_artifact_registry(is_real=True)
    artifact_registry.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=component_version, expandvars=expandvars, collect_inds=[1], disable_inds=[1])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=9
Creating /home/misha/dev/mine/neurolab/17_rl/17c_ppo_trxl_memory_mp_01-launch9.ipynb
Created launch notebook "/home/misha/dev/mine/neurolab/17_rl/17c_ppo_trxl_memory_mp_01-launch9.ipynb"


## Optuna (model selection)

### Templates

In [ ]:
# @launchit.disable
# @launchit.collect_3
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        case 1:
            HP = Hyperparameters()
            HP.random_seed = 42
            assert False
        case _:
            assert False, f'Unsupported {study_serial=}'            

### Unleash

In [ ]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL:
            return ['minimize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 1
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 1
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=4, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

In [ ]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")